In [ ]:
# ========================================
# 1. MOUNT GOOGLE DRIVE
# ========================================
from google.colab import drive
drive.mount('/content/drive')

# Verify your files exist
!ls "/content/drive/MyDrive/ODIR-5K/"

# ========================================
# 2. INSTALL DEPENDENCIES
# ========================================
!pip install -q torch torchvision torchaudio
!pip install -q pandas openpyxl scikit-learn ptflops timm
import warnings
warnings.filterwarnings("ignore")


Mounted at /content/drive
 best_model.pth      data.xlsx	       Training_Images
 data_updated.xlsx  'Testing Images'


In [ ]:
# ========================================================
# LIGHTHYBRIDNET FOR ODIR-5K WITH OVERSAMPLING
# ========================================================

!pip install -q timm albumentations tqdm fvcore imbalanced-learn --upgrade
import os
import shutil
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, roc_auc_score, cohen_kappa_score
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from tqdm.auto import tqdm
import timm
import random
from torch.amp import autocast, GradScaler
from fvcore.nn import FlopCountAnalysis
from collections import Counter

# ------------------- Reproducibility -------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# ------------------- Mount Drive -------------------
from google.colab import drive
drive.mount('/content/drive')

# ------------------- Paths -------------------
CSV_PATH = '/content/drive/MyDrive/ODIR-5K/full_df.csv'
IMG_SOURCE_DIR = '/content/drive/MyDrive/ODIR-5K/preprocessed_images'
LOCAL_IMG_PATH = '/content/images'
SAVE_DIR = '/content/drive/MyDrive/ODIR-5K/new_models'
os.makedirs(SAVE_DIR, exist_ok=True)
SAVE_PATH = os.path.join(SAVE_DIR, 'best_lighthybrid_odir_smote_AUC.pth')

if os.path.exists(LOCAL_IMG_PATH):
    shutil.rmtree(LOCAL_IMG_PATH)
os.makedirs(LOCAL_IMG_PATH, exist_ok=True)

print("Copying images to local disk for faster loading...")
for f in tqdm(os.listdir(IMG_SOURCE_DIR)):
    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
        shutil.copy2(os.path.join(IMG_SOURCE_DIR, f), LOCAL_IMG_PATH)
print("Copy complete!")

# ------------------- Load and prepare dataframe -------------------
df = pd.read_csv(CSV_PATH)
LABELS = ['N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']
left_df = df[['Left-Fundus'] + LABELS].rename(columns={'Left-Fundus': 'filename'})
right_df = df[['Right-Fundus'] + LABELS].rename(columns={'Right-Fundus': 'filename'})
full_df = pd.concat([left_df, right_df], ignore_index=True)
full_df['path'] = full_df['filename'].apply(lambda x: os.path.join(LOCAL_IMG_PATH, x))
full_df = full_df[full_df['path'].apply(os.path.exists)].reset_index(drop=True)
print(f"Total valid images: {len(full_df)}")

# Train/val split with stratification (using primary label)
train_df, val_df = train_test_split(
    full_df,
    test_size=0.2,
    random_state=42,
    stratify=full_df[LABELS].idxmax(axis=1)
)

# ------------------- Replication-based Oversampling (SMOTE-inspired balancing) -------------------
# We balance the primary disease classes by replicating minority primary class images
train_df['primary_label'] = train_df[LABELS].idxmax(axis=1)

primary_counts = Counter(train_df['primary_label'])
print("Primary class distribution before oversampling:", primary_counts)

max_count = max(primary_counts.values())
oversampled_rows = []

for _, row in train_df.iterrows():
    primary = row['primary_label']
    target_count = max_count
    current_count = primary_counts[primary]
    replication_factor = max(1, target_count // current_count)
    for _ in range(replication_factor):
        oversampled_rows.append(row.copy())

train_df = pd.DataFrame(oversampled_rows).reset_index(drop=True)
train_df = train_df.drop(columns=['primary_label'])

print(f"Train samples after replication oversampling: {len(train_df)}")

# ------------------- Augmentations -------------------
train_transforms = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.Rotate(limit=30, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.6),
    A.CLAHE(clip_limit=2.0, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, border_mode=0, p=0.6),
    A.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.5),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

val_transforms = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# ------------------- Dataset -------------------
class ODIRDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = Image.open(row['path']).convert('RGB')
            img = np.array(img)
            if self.transform:
                img = self.transform(image=img)['image']
            labels = row[LABELS].values.astype(np.float32)
            return img, torch.from_numpy(labels)
        except Exception as e:
            print(f"Error loading {row['path']}: {e}")
            return torch.zeros(3, 224, 224), torch.zeros(8)

train_dataset = ODIRDataset(train_df, train_transforms)
val_dataset = ODIRDataset(val_df, val_transforms)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

# ------------------- Model: LightHybridNet -------------------
class LightHybridNet(nn.Module):
    def __init__(self, num_classes=8, dropout=0.5):
        super().__init__()
        self.mobilenet = timm.create_model('mobilenetv3_large_100.ra_in1k', pretrained=True, num_classes=0)
        self.effnet = timm.create_model('efficientnet_b1', pretrained=True, num_classes=0)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(1280 + 1280, num_classes)

    def forward(self, x):
        f1 = self.mobilenet(x)
        f2 = self.effnet(x)
        f = torch.cat([f1, f2], dim=1)
        f = self.dropout(f)
        return self.head(f)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = LightHybridNet().to(DEVICE)

# ------------------- Model Stats -------------------
total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"\nModel: LightHybridNet")
print(f"Total Parameters: {total_params:.2f} Million")

dummy_input = torch.randn(1, 3, 224, 224).to(DEVICE)
flops = FlopCountAnalysis(model, dummy_input)
print(f"Estimated FLOPs: {flops.total() / 1e9:.3f} GFLOPs\n")

# ------------------- Training setup -------------------
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
scaler = GradScaler('cuda')
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=3e-4,
    total_steps=len(train_loader) * 25,
    pct_start=0.1,
    anneal_strategy='cos'
)

# prediction
def get_smart_preds(probs, thresh=0.5):
    preds = np.zeros_like(probs)
    for i, p in enumerate(probs):
        if p[0] > 0.8:
            preds[i, 0] = 1
        else:
            mask = p > thresh
            if mask.sum() == 0:
                preds[i, np.argmax(p)] = 1
            else:
                preds[i] = mask.astype(int)
    return preds

# ------------------- Validation Function -------------------
def validate(model, loader):
    model.eval()
    all_probs, all_targets = [], []
    with torch.no_grad():
        for img, lbl in loader:
            img = img.to(DEVICE)
            logits = model(img)
            probs = torch.sigmoid(logits)
            all_probs.append(probs.cpu().numpy())
            all_targets.append(lbl.numpy())
    probs = np.vstack(all_probs)
    targets = np.vstack(all_targets)
    preds = get_smart_preds(probs, thresh=0.5)
    weighted_f1 = f1_score(targets, preds, average='weighted', zero_division=0)
    kappa = cohen_kappa_score(targets.flatten(), preds.flatten())
    try:
        auc_macro = roc_auc_score(targets, probs, average='macro')
    except ValueError:
        auc_macro = float('nan')
    auc_per_class = []
    for i in range(targets.shape[1]):
        if np.sum(targets[:, i]) > 0 and np.sum(1 - targets[:, i]) > 0:
            auc_per_class.append(roc_auc_score(targets[:, i], probs[:, i]))
        else:
            auc_per_class.append(float('nan'))
    return weighted_f1, probs, targets, preds, auc_macro, auc_per_class, kappa

# ------------------- Training Loop -------------------
EPOCHS = 25
best_f1 = 0.0
print("Starting training with replication-based oversampling (SMOTE-inspired)...\n")
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for img, lbl in tqdm(train_loader, desc=f"Epoch {epoch+1:02d} [Train]", leave=False):
        img, lbl = img.to(DEVICE), lbl.to(DEVICE)
        optimizer.zero_grad()
        with autocast('cuda'):
            logits = model(img)
            loss = criterion(logits, lbl)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        train_loss += loss.item()
    val_f1, _, _, _, auc_macro, _, val_kappa = validate(model, val_loader)
    print(f"Epoch {epoch+1:02d} | Loss: {train_loss/len(train_loader):.4f} | "
          f"Val Weighted F1: {val_f1:.4f} | "
          f"Val AUC (macro): {auc_macro:.4f} | "
          f"Val Kappa: {val_kappa:.4f}")
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), SAVE_PATH)
        print(f" → New best model saved! F1 = {best_f1:.4f}")

print(f"\nTraining finished! Best Weighted F1: {best_f1:.4f}")

# ------------------- Final Evaluation -------------------
print("\nLoading best model for final evaluation...")
model.load_state_dict(torch.load(SAVE_PATH))
final_f1, val_probs, val_targets, val_preds, final_auc_macro, final_auc_per_class, final_kappa = validate(model, val_loader)

print("\n" + "="*80)
print("FINAL RESULTS - ODIR-5K VALIDATION SET (REPLICATION OVERSAMPLING)")
print("="*80)
print(f"Total Parameters : {total_params:.2f} M")
print(f"Estimated FLOPs : {flops.total() / 1e9:.3f} GFLOPs")
print(f"Weighted F1-Score : {final_f1:.4f}")
print(f"AUC (macro) : {final_auc_macro:.4f}")
print(f"Cohen's Kappa : {final_kappa:.4f}")
print("\nPer-class AUC:")
for label, auc in zip(LABELS, final_auc_per_class):
    print(f" {label}: {auc:.4f}" if not np.isnan(auc) else f" {label}: N/A")
print("\nPer-class Classification Report:")
print(classification_report(val_targets, val_preds, target_names=LABELS, digits=4, zero_division=0))
print(f"\nBest model saved at:\n{SAVE_PATH}")
print("="*80)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copying images to local disk for faster loading...


  0%|          | 0/6394 [00:00<?, ?it/s]

Copy complete!
Total valid images: 12460
Primary class distribution before oversampling: Counter({'D': 3327, 'N': 3315, 'O': 1336, 'G': 537, 'C': 506, 'A': 423, 'M': 386, 'H': 138})
Train samples after replication oversampling: 24933


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipython-input-2241695286.py:111: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.5),
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/22.1M [00:00<?, ?B/s]

...

Epoch 23 | Loss: 0.0040 | Val Weighted F1: 0.9101 | Val AUC (macro): 0.9830 | Val Kappa: 0.8955


Epoch 24 [Train]:   0%|          | 0/390 [00:00<?, ?it/s]

Epoch 24 | Loss: 0.0041 | Val Weighted F1: 0.9117 | Val AUC (macro): 0.9830 | Val Kappa: 0.8973


Epoch 25 [Train]:   0%|          | 0/390 [00:00<?, ?it/s]

Epoch 25 | Loss: 0.0040 | Val Weighted F1: 0.9120 | Val AUC (macro): 0.9835 | Val Kappa: 0.8977

Training finished! Best Weighted F1: 0.9126

Loading best model for final evaluation...

FINAL RESULTS - ODIR-5K VALIDATION SET (REPLICATION OVERSAMPLING)
Total Parameters : 10.74 M
Estimated FLOPs : 0.853 GFLOPs
Weighted F1-Score : 0.9126
AUC (macro) : 0.9829
Cohen's Kappa : 0.8985

Per-class AUC:
 N: 0.9835
 D: 0.9807
 G: 0.9838
 C: 0.9932
 A: 0.9959
 H: 0.9778
 M: 0.9941
 O: 0.9543

Per-class Classification Report:
              precision    recall  f1-score   support

           N     0.8924    0.9420    0.9166       828
           D     0.9267    0.9123    0.9194       832
           G     0.9463    0.8924    0.9186       158
           C     0.9671    0.9484    0.9577       155
           A     0.9839    0.8841    0.9313       138
           H     0.9577    0.8293    0.8889        82
           M     0.9533    0.9358    0.9444       109
           O     0.9308    0.8288    0.8768     